# Working with *GAMESS* Files in *Python*

The results file may contain **thousands of lines** of text. How can we find the useful bits? In this book we use ***Python*** in ***Jupyter* notebooks**. A few lines of *Python* can find any line in a file, turn it into **numbers**, write new input files and run a whole **series** of jobs. This page introduces the handful of tools that we will use in every chapter.

The examples use the log file of the water test job from the [setup page](1_Setup.ipynb). If that file is missing, the settings cell below runs the job (it takes a few seconds).

### Settings

In [1]:
# ===== SETTINGS =========================================================
# Your GAMESS location is set ONCE, in gamess_settings.py (next to this
# notebook). Edit that file, save it, and restart the kernel.
from pathlib import Path
import gamess_tools as gt

INPUTS = Path("inputs")     # input files that come with the book
RUNS = Path("runs")         # where log files and new input files are written
RUNS.mkdir(exist_ok=True)
# ========================================================================

log = RUNS / "water_test.log"
if not gt.terminated_normally(log):
    log = gt.run_gamess(INPUTS / "water_test.inp")
print("using", log)

using runs/water_test.log


## The Notebook

A *Jupyter* notebook is a page of text with **cells** of code in it. Click on a cell and press `Shift`+`Return` to run it; the result appears underneath. Each cell can use the **variables** made by the cells that ran before it, so run the cells from the top down. If things get confused, choose `Kernel` &rarr; `Restart` and run the cells again from the top.

Every notebook in this book starts with a **settings cell** like the one above. It loads `gamess_tools.py`, the book's small helper module, under the short name `gt`. Every function that starts with `gt.` comes from that module.

## Folders and Files

A notebook runs **in its own folder**, the folder that holds the notebook file. Paths are written relative to that folder with `Path` from the *Python* standard library. `Path.cwd()` shows where we are, and `.glob()` lists the files that match a pattern. The `*` is a **wildcard** that matches any text.

In [2]:
print(Path.cwd())
print(sorted(p.name for p in INPUTS.glob("water*.inp")))

/Users/blink/github/UsingGAMESSv2
['water.inp', 'water_6311_Z_E.inp', 'water_AM1_U.inp', 'water_Z_AM1.inp', 'water_template.inp', 'water_test.inp']


A path also knows its **parts**: the file name, the name without the extension (the "stem"), the extension and the folder it is in. We will often use the stem of an input file to name the files that belong to the same job.

In [3]:
print(log.name, "|", log.stem, "|", log.suffix, "|", log.parent)

water_test.log | water_test | .log | runs


## Getting Help

`help()` shows the **documentation** of any *Python* function, including the ones in `gamess_tools.py`. In *JupyterLab* you can also type a question mark after a name, for example `gt.grep?`, and run the cell. Between `help()` and the internet you can **learn more** about any of the tools on this page.

In [4]:
help(gt.grep)

Help on function grep in module gamess_tools:

grep(text, file)
    Return the lines of `file` that contain `text` (like: grep "text" file).



## Reading a File

`read_text()` reads a whole file into one long string and `splitlines()` cuts it into a **list** of lines. Items in a list are counted from 0, and negative numbers count from the end: `lines[-5:]` is the last five lines. These last lines are not from *GAMESS* itself. They are written by the *rungms* script after *GAMESS* has finished: a few housekeeping messages, the date and the time the job took.

In [5]:
lines = log.read_text().splitlines()
print(len(lines), "lines")
print("\n".join(lines[-5:]))

1297 lines
ls: No match.
ls: No match.
ls: No match.
Sat Nov  7 14:19:14 AST 2026
0.118u 0.201s 0:01.32 23.4%	0+0k 0+0io 420pf+0w


## Finding Lines

Reading hundreds of lines by eye is slow. `gt.grep()` returns only the lines of a file that **contain** a piece of text. Did the job finish? *GAMESS* prints `TERMINATED NORMALLY` when it does.

In [6]:
print(gt.grep("TERMINATED", log))

[' EXECUTION OF GAMESS TERMINATED NORMALLY 14:19:13 07-NOV-2026    ']


Each step of a geometry optimization is reported on a line that contains `NSERCH`.

In [7]:
for line in gt.grep("NSERCH", log):
    print(line)

 BEGINNING GEOMETRY SEARCH POINT NSERCH=   0 ...
          NSERCH=  0     ENERGY=     -74.9611354
 NSERCH:   0  E=      -74.9611354038  GRAD. MAX=  0.0325054  R.M.S.=  0.0180583
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   1 ...
          NSERCH=  1     ENERGY=     -74.9634131
 NSERCH:   1  E=      -74.9634131199  GRAD. MAX=  0.0418289  R.M.S.=  0.0171359
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   2 ...
          NSERCH=  2     ENERGY=     -74.9658650
 NSERCH:   2  E=      -74.9658650131  GRAD. MAX=  0.0040888  R.M.S.=  0.0022627
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   3 ...
          NSERCH=  3     ENERGY=     -74.9658956
 NSERCH:   3  E=      -74.9658956331  GRAD. MAX=  0.0016287  R.M.S.=  0.0009369
 BEGINNING GEOMETRY SEARCH POINT NSERCH=   4 ...
          NSERCH=  4     ENERGY=     -74.9659012
 NSERCH:   4  E=      -74.9659012164  GRAD. MAX=  0.0000273  R.M.S.=  0.0000127


That is **every** line that includes `NSERCH`, and some of them do not hold an energy. The result of one step can be **fed** into the next: here we keep only the lines that also contain `GRAD`. The square brackets make a new list from the old one (a "list comprehension"), and `[-1]` picks the last item, the final step of the optimization.

In [8]:
steps = [line for line in gt.grep("NSERCH", log) if "GRAD" in line]
for line in steps:
    print(line)

last = steps[-1]
print("\nlast step:", last)

 NSERCH:   0  E=      -74.9611354038  GRAD. MAX=  0.0325054  R.M.S.=  0.0180583
 NSERCH:   1  E=      -74.9634131199  GRAD. MAX=  0.0418289  R.M.S.=  0.0171359
 NSERCH:   2  E=      -74.9658650131  GRAD. MAX=  0.0040888  R.M.S.=  0.0022627
 NSERCH:   3  E=      -74.9658956331  GRAD. MAX=  0.0016287  R.M.S.=  0.0009369
 NSERCH:   4  E=      -74.9659012164  GRAD. MAX=  0.0000273  R.M.S.=  0.0000127

last step:  NSERCH:   4  E=      -74.9659012164  GRAD. MAX=  0.0000273  R.M.S.=  0.0000127


## From Text to Numbers

A line of text is not yet a number that we can plot or use in a calculation. `split()` breaks a line into **words** at the spaces, and `float()` turns a word into a number. In these lines the energy is the fourth word, which is item 3 because we count from 0.

In [9]:
words = last.split()
print(words)

energy = float(words[3])
print(f"final energy = {energy:.10f} hartree")

['NSERCH:', '4', 'E=', '-74.9659012164', 'GRAD.', 'MAX=', '0.0000273', 'R.M.S.=', '0.0000127']
final energy = -74.9659012164 hartree


The helper `gt.nserch_energies()` does exactly this for every step and returns a list of `(step, energy)` pairs. Many of the `gt.` functions work this way: find the lines, split them, and turn the words into numbers.

In [10]:
for step, energy in gt.nserch_energies(log):
    print(step, energy)

0 -74.9611354038
1 -74.9634131199
2 -74.9658650131
3 -74.9658956331
4 -74.9659012164


## Writing a File

`write_text()` writes a string into a file, and **replaces** the file if it already exists. `"\n".join(...)` glues a list of lines together with a line break between them. This is how the notebooks save a table of results for later.

In [11]:
summary = RUNS / "water_test_energies.txt"
table = [f"{step:3d} {energy:.10f}" for step, energy in gt.nserch_energies(log)]
summary.write_text("\n".join(table) + "\n")
print(summary.read_text())

  0 -74.9611354038
  1 -74.9634131199
  2 -74.9658650131
  3 -74.9658956331
  4 -74.9659012164



## Making Files from a Template

Often we want many input files that differ in just **one** value: a basis set, a bond length or a torsion angle. We write one **template** file with a placeholder word where the value goes. In this template the placeholder `NUMBER` sets the size of the STO-*n*G basis set.

In [12]:
template = INPUTS / "water_template.inp"
print(template.read_text())

! Water, RHF/STO-nG geometry optimisation.
! Template: the placeholder after NGAUSS= is replaced by 2, 3 or 6 (STO-2G, STO-3G, STO-6G).
 $CONTRL SCFTYP=RHF RUNTYP=OPTIMIZE MAXIT=30 MULT=1 ICHARG=0 COORD=CART $END
 $SYSTEM TIMLIM=10 MWORDS=10 $END
 $BASIS GBASIS=STO NGAUSS=NUMBER $END
 $STATPT OPTTOL=0.0001 NSTEP=100 $END
 $DATA
Water, STO-nG basis
CNV 2

O     8.0   0.0   0.0   0.0
H     1.0  -0.7   0.0  -0.7
H     1.0   0.7   0.0  -0.7
 $END



`gt.substitute()` copies the template into a new file and **replaces** the placeholder on the way. The replacements are a dictionary: the text to find, and the text to put in its place.

In [13]:
new_input = RUNS / "water_sto3g.inp"
gt.substitute(template, new_input, {"NUMBER": "3"})
print(gt.grep("$BASIS", new_input))

[' $BASIS GBASIS=STO NGAUSS=3 $END']


## Doing It Again and Again

A `for` loop repeats the same steps for each item in a list. Here one loop does the whole job for three basis sets: **make** an input file from the template, **run** it with *GAMESS*, and **collect** the final energy. The three jobs take a few seconds.

In [14]:
results = []
for n in (2, 3, 6):
    new_input = RUNS / f"water_sto{n}g.inp"
    gt.substitute(template, new_input, {"NUMBER": str(n)})
    log_n = gt.run_gamess(new_input)
    results.append((f"STO-{n}G", gt.final_energy(log_n)))

print()
for basis, energy in results:
    print(f"{basis:8s} E = {energy:.10f} hartree")

water_sto2g: finished normally (water_sto2g.log)


water_sto3g: finished normally (water_sto3g.log)


water_sto6g: finished normally (water_sto6g.log)

STO-2G   E = -72.7440788079 hartree
STO-3G   E = -74.9659012164 hartree
STO-6G   E = -75.6812004086 hartree


The energy goes **down** as the basis set gets bigger. More Gaussian functions describe each atomic orbital more accurately, and a better description of the orbitals gives a lower energy. We will meet this idea, the **level of theory**, again in the [next chapter](2_SmallMolecules.ipynb).

The same pattern of a template, a loop and a list of results builds the rotation profile of butane, the bond-rotation scan of butadiene and the reaction coordinate of the Diels-Alder reaction later in the book.

## Under the Hood

The `gt.` functions are not magic. Each one is only a few lines long and uses the same tools as this page: `Path`, lists, `split()`, `float()` and loops. `inspect.getsource()` prints the code of a function. Here is `gt.grep()`:

In [15]:
import inspect
print(inspect.getsource(gt.grep))

def grep(text, file):
    """Return the lines of `file` that contain `text` (like: grep "text" file)."""
    with open(file, errors="replace") as f:
        return [line.rstrip("\n") for line in f if text in line]



The [helper code page](1E_Helper_Code.md) walks through every function in `gamess_tools.py`. You will still need the **terminal** to install and compile *GAMESS* (see the [setup page](1_Setup.ipynb)), but once *GAMESS* works, everything else happens in the notebooks.

## Summary

| Task | *Python* |
| :--- | :--- |
| Where am I? | `Path.cwd()` |
| List files | `sorted(Path("runs").glob("*.log"))` |
| Get help | `help(gt.grep)` or `gt.grep?` |
| Read a file | `Path("file").read_text().splitlines()` |
| Find lines | `gt.grep("NSERCH", log)` |
| Keep some of them | `[line for line in lines if "GRAD" in line]` |
| The last one | `lines[-1]` |
| Text to numbers | `line.split()` and `float()` |
| Write a file | `Path("file.txt").write_text(text)` |
| Fill in a template | `gt.substitute(template, new_file, {"NUMBER": "3"})` |
| Run *GAMESS* | `gt.run_gamess("inputs/file.inp")` |
| Repeat | `for n in (2, 3, 6): ...` |

## AI Statement

This page replaces the original page `1A_Unix.md`, which taught the *Unix* shell and the tools *grep*, *sed* and *awk*. At the author's request it was rewritten by Claude (Anthropic) as an introduction to the *Python* tools that the notebooks use, so that the book reads as written for *Python*.

- **Original text kept (about 5 %).** The opening question ("The results file may contain thousands of lines of text. How can we find the useful bits?") and the order of the topics (finding your way around, getting help, searching files, turning columns into numbers, writing files, substituting text and scripting a series of jobs) follow the author's page.
- **Written by AI.** All other text, all code cells, the STO-*n*G basis-set loop and the summary table. The file keeps its name, `1A_Unix.ipynb`, so that links from other pages keep working.